# EECE 6544 — Assignment #02
## Predicting Monthly Charges from Add-On Services (Multivariable Linear Regression)

In [1]:
!pip install scikit-learn

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## 1. Load and inspect the data
Read `telco.csv` , confirm the target is `target` (MonthlyCharges, continuous), and verify the service columns are clean 0/1 indicators with no missing values.

In [3]:
df = pd.read_csv("telco.csv")
print("Shape:", df.shape)
df.head()

Shape: (7043, 24)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,target,TotalCharges,Churn,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,0,0,1,0,...,29.85,29.85,0,0,0,0,0,0,1,0
1,1,0,0,0,34,1,0,1,0,1,...,56.95,1889.5,0,0,0,1,0,0,0,1
2,1,0,0,0,2,1,0,1,1,0,...,53.85,108.15,1,0,0,0,0,0,0,1
3,1,0,0,0,45,0,0,1,0,1,...,42.30,1840.75,0,0,0,1,0,0,0,0
4,0,0,0,0,2,1,0,0,0,0,...,70.70,151.65,1,1,0,0,0,0,1,0


In [4]:
service_cols = [
    "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "InternetService_Fiber optic", "InternetService_No",
]

# Target is continuous (MonthlyCharges)
print("Target column 'target' (MonthlyCharges):")
print(df["target"].describe())

# Service columns are clean 0/1 indicators with no missing values
print("\nMissing values in service columns:", df[service_cols].isna().sum().sum())
print("Missing values in target:", df["target"].isna().sum())
print("\nUnique values per service column:")
for col in service_cols:
    print(f"  {col}: {sorted(df[col].unique())}")

Target column 'target' (MonthlyCharges):
count    7043.000000
mean       64.761692
std        30.090047
min        18.250000
25%        35.500000
50%        70.350000
75%        89.850000
max       118.750000
Name: target, dtype: float64

Missing values in service columns: 0
Missing values in target: 0

Unique values per service column:
  PhoneService: [0, 1]
  MultipleLines: [0, 1]
  OnlineSecurity: [0, 1]
  OnlineBackup: [0, 1]
  DeviceProtection: [0, 1]
  TechSupport: [0, 1]
  StreamingTV: [0, 1]
  StreamingMovies: [0, 1]
  InternetService_Fiber optic: [0, 1]
  InternetService_No: [0, 1]


## 2. Define the variables
Target = `MonthlyCharges` (the `target` column). Predictors = the ten service indicators.

In [5]:
X = df[service_cols]
y = df["target"]  # MonthlyCharges
print("Predictors:", list(X.columns))
print("X shape:", X.shape, "| y shape:", y.shape)

Predictors: ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'InternetService_Fiber optic', 'InternetService_No']
X shape: (7043, 10) | y shape: (7043,)


## 3. Split the data
Hold out a test set (80/20) with a fixed random seed so results are reproducible.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])

Train size: 5634 | Test size: 1409


## 4. Fit a multivariable linear regression
Train the model $\hat{y} = w_0 + \sum_i w_i x_i$ on the training set.

In [7]:
model = LinearRegression()
model.fit(X_train, y_train)
print("Model trained.")

Model trained.


## 5. Report the parameters
The intercept is the base-plan charge; each coefficient is the dollar contribution of that service.

In [8]:
print(f"Intercept (base-plan charge): ${model.intercept_:.2f}\n")
coefs = pd.Series(model.coef_, index=service_cols).sort_values(ascending=False)
print("Coefficients (dollar contribution of each service):")
for name, w in coefs.items():
    print(f"  {name:<30} ${w:+.2f}")

Intercept (base-plan charge): $24.97

Coefficients (dollar contribution of each service):
  InternetService_Fiber optic    $+24.95
  PhoneService                   $+20.04
  StreamingTV                    $+9.98
  StreamingMovies                $+9.95
  OnlineSecurity                 $+5.05
  TechSupport                    $+5.03
  MultipleLines                  $+5.01
  DeviceProtection               $+5.01
  OnlineBackup                   $+4.98
  InternetService_No             $-25.05


## 6. Evaluate on the test set
Report R², mean absolute error (MAE), and RMSE.

In [9]:
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R²   : {r2:.4f}")
print(f"MAE  : ${mae:.2f}")
print(f"RMSE : ${rmse:.2f}")

R²   : 0.9988
MAE  : $0.79
RMSE : $1.05


## 7. Compare to a baseline
Baseline = the original single-variable model that only knows the *count* of add-ons. Both models use the same split.

In [10]:
# Single predictor: how many services the customer has
count_train = X_train.sum(axis=1).to_frame("service_count")
count_test = X_test.sum(axis=1).to_frame("service_count")

baseline = LinearRegression()
baseline.fit(count_train, y_train)
y_pred_base = baseline.predict(count_test)

r2_base = r2_score(y_test, y_pred_base)
mae_base = mean_absolute_error(y_test, y_pred_base)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))

comparison = pd.DataFrame({
    "R²": [r2_base, r2],
    "MAE ($)": [mae_base, mae],
    "RMSE ($)": [rmse_base, rmse],
}, index=["Baseline (count of add-ons)", "Multivariable (which add-ons)"])
comparison.round(3)

,R²,MAE ($),RMSE ($)
Baseline (count of add-ons),0.701,14.237,16.455
Multivariable (which add-ons),0.999,0.789,1.050


## 8. Interpret the results
Each weight ≈ the monthly price of that service, in plain business terms.

In [11]:
print(f"Base plan (no add-ons): about ${model.intercept_:.2f}/month.\n")
print("Learned monthly price of each add-on:")
for name, w in coefs.items():
    print(f"  {name:<30} adds about ${w:.2f}/month")

print(f"\nMost expensive add-on : {coefs.idxmax()} (${coefs.max():.2f})")
print(f"Cheapest add-on       : {coefs.drop('InternetService_No').idxmin()} (${coefs.drop('InternetService_No').min():.2f})")

fiber_streaming = (
    coefs["InternetService_Fiber optic"] + coefs["StreamingTV"] + coefs["StreamingMovies"]
)
print(f"\nAdding fiber internet + both streaming services raises the bill by about ${fiber_streaming:.2f}/month.")
print("\nNote: 'InternetService_No' has a negative weight — having no internet service")
print("lowers the bill, which matches intuition (internet is part of the base charge here).")

Base plan (no add-ons): about $24.97/month.

Learned monthly price of each add-on:
  InternetService_Fiber optic    adds about $24.95/month
  PhoneService                   adds about $20.04/month
  StreamingTV                    adds about $9.98/month
  StreamingMovies                adds about $9.95/month
  OnlineSecurity                 adds about $5.05/month
  TechSupport                    adds about $5.03/month
  MultipleLines                  adds about $5.01/month
  DeviceProtection               adds about $5.01/month
  OnlineBackup                   adds about $4.98/month
  InternetService_No             adds about $-25.05/month

Most expensive add-on : InternetService_Fiber optic ($24.95)
Cheapest add-on       : OnlineBackup ($4.98)

Adding fiber internet + both streaming services raises the bill by about $44.88/month.

Note: 'InternetService_No' has a negative weight — having no internet service
lowers the bill, which matches intuition (internet is part of the base charge h

## 9. Make a prediction
Expected monthly charge for a new customer's chosen bundle — e.g. phone service, multiple lines, and tech support.

In [12]:
new_customer = pd.DataFrame([{
    "PhoneService": 1,
    "MultipleLines": 1,
    "OnlineSecurity": 0,
    "OnlineBackup": 0,
    "DeviceProtection": 0,
    "TechSupport": 1,
    "StreamingTV": 0,
    "StreamingMovies": 0,
    "InternetService_Fiber optic": 0,
    "InternetService_No": 0,
}])[service_cols]

expected_charge = model.predict(new_customer)[0]
print("Bundle: phone service + multiple lines + tech support (DSL internet)")
print(f"Expected monthly charge: ${expected_charge:.2f}")

Bundle: phone service + multiple lines + tech support (DSL internet)
Expected monthly charge: $55.05
